# *数智学院课程 — 《数据挖掘》*

## 第 §10 章：异常检测与模式发现

### 📝 课后作业与拓展

> 教师: 谢海华  
> 学期: 2026 春季  
> 作业完成版


## 1. 概念理解题

### （1）关于异常

**点异常**：单个样本点相对于整体数据分布明显偏离。例如，一个平时每天消费几十元的学生，某天银行卡突然支出 10000 元。

**群体异常**：单个点看起来不一定异常，但一组样本共同出现时表现异常。例如，某个账号在 10 分钟内连续访问几十个商品页、频繁加入购物车又取消，单次点击并不异常，但组合行为异常。

**情境异常**：行为是否异常取决于具体上下文。比如凌晨 3 点在宿舍楼下跑步，对备赛运动员可能正常；但如果一个平时规律作息、从不夜跑的人突然连续几天凌晨外出跑步，就可能是异常行为。

一个身边例子：期末周每天学习到凌晨 2 点是相对正常的，因为考试压力集中；但普通周连续熬夜到凌晨 2 点刷手机、第二天旷课，就更像异常行为。

### （2）Isolation Forest 与 LOF 对比

**Isolation Forest** 的核心思想是：异常点通常更“孤立”，更容易被随机切分隔离出来，因此在随机树中路径长度更短。它主要从“被隔离的难易程度”判断异常。

**LOF（Local Outlier Factor）** 的核心思想是：比较一个点附近的局部密度和其邻居附近的局部密度。如果某点的局部密度显著低于邻居，则它更可能是异常。它主要从“局部密度相对差异”判断异常。

当数据存在明显的**局部密度差异**时，LOF 通常更适合，因为它不是只看全局稀疏程度，而是看一个点相对于周围邻居是否“突兀”。Isolation Forest 更适合高维、大规模、异常点较孤立的场景。

### （3）关于序列

**时间序列**强调同一个指标随时间变化的数值序列，例如每天气温、每小时访问量、每分钟交易金额。它的重点是趋势、周期性、波动和异常点。

**行为序列**强调用户或对象按时间发生的一串离散行为，例如 `view_home → view_product → add_to_cart → checkout`。它的重点是行为转移、路径模式和流程瓶颈。

三个时间序列指标例子：

1. 某网站每小时访问量。
2. 某股票每天收盘价。
3. 某学生每天学习时长。

一个行为序列数据例子：某电商用户从进入首页、浏览商品、加入购物车、结算到离开的完整点击路径。

### （4）时间序列中的趋势、季节性和噪声

在时间序列中区分**趋势、季节性和噪声**，是为了把长期变化、周期性规律和随机扰动分开。趋势反映长期上升或下降，例如用户量持续增长；季节性反映固定周期内重复出现的规律，例如周末访问量更高；噪声是不稳定、不可解释的随机波动。如果不拆开看，模型可能把周期性误判为异常，或者把随机波动误认为趋势。

**移动平均**通过取一个时间窗口内的均值来平滑短期波动，让长期方向更清楚。窗口越大，曲线越平滑，噪声越少，但对突发变化的反应也越慢；窗口越小，越敏感，但也更容易受随机波动干扰。

## 2. 代码实践

### 2.1 欺诈 / 异常行为识别

这里构造一个模拟交易数据集。正常交易占多数，欺诈交易占少数。训练时，IsolationForest 只使用正常样本；评估时，在包含正常与欺诈的测试集上计算分类指标。然后用同一份标注数据训练 LogisticRegression 和 RandomForest 作对比。

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

np.random.seed(42)

# -----------------------------
# 1. 构造模拟交易数据
# -----------------------------
n_normal = 1200
n_fraud = 80

normal = pd.DataFrame({
    "amount": np.random.lognormal(mean=3.2, sigma=0.55, size=n_normal),
    "hour": np.random.choice(range(8, 24), size=n_normal, p=np.ones(16)/16),
    "device_change": np.random.binomial(1, 0.08, size=n_normal),
    "num_txn_1h": np.random.poisson(1.5, size=n_normal),
    "distance_from_usual_km": np.random.exponential(scale=4, size=n_normal),
    "label": 0
})

fraud = pd.DataFrame({
    "amount": np.random.lognormal(mean=4.5, sigma=0.75, size=n_fraud),
    "hour": np.random.choice(range(0, 24), size=n_fraud),
    "device_change": np.random.binomial(1, 0.65, size=n_fraud),
    "num_txn_1h": np.random.poisson(5.0, size=n_fraud),
    "distance_from_usual_km": np.random.exponential(scale=35, size=n_fraud),
    "label": 1
})

transactions = pd.concat([normal, fraud], ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)
features = ["amount", "hour", "device_change", "num_txn_1h", "distance_from_usual_km"]

X = transactions[features]
y = transactions["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)

print("数据规模:", transactions.shape)
print("欺诈比例:", round(y.mean(), 4))
transactions.head()

数据规模: (1280, 6)
欺诈比例: 0.0625


,amount,hour,device_change,num_txn_1h,distance_from_usual_km,label
0,239.114567,23,1,4,30.716368,1
1,20.692048,23,1,1,18.510700,0
2,15.653357,11,0,1,1.223240,0
3,36.207402,14,0,4,5.874746,0
4,24.887112,12,0,2,1.219005,0


In [2]:
# -----------------------------
# 2. IsolationForest：仅用正常样本训练
# -----------------------------
X_train_normal = X_train[y_train == 0]

iso_model = make_pipeline(
    StandardScaler(),
    IsolationForest(
        n_estimators=200,
        contamination=y_train.mean(),
        random_state=42
    )
)
iso_model.fit(X_train_normal)

# IsolationForest 输出：-1 表示异常，1 表示正常
iso_raw_pred = iso_model.predict(X_test)
iso_pred = (iso_raw_pred == -1).astype(int)

# decision_function 越大越正常，因此取负值作为异常分数
iso_score = -iso_model.decision_function(X_test)

print("IsolationForest 评估结果：")
print(classification_report(y_test, iso_pred, digits=3))
print("ROC-AUC:", round(roc_auc_score(y_test, iso_score), 3))
print("混淆矩阵:\n", confusion_matrix(y_test, iso_pred))

IsolationForest 评估结果：
              precision    recall  f1-score   support

           0      1.000     0.933     0.966       360
           1      0.500     1.000     0.667        24

    accuracy                          0.938       384
   macro avg      0.750     0.967     0.816       384
weighted avg      0.969     0.938     0.947       384

ROC-AUC: 1.0
混淆矩阵:
 [[336  24]
 [  0  24]]


In [3]:
# -----------------------------
# 3. 有监督模型：LogisticRegression / RandomForest
# -----------------------------
log_reg = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
)
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=6,
    class_weight="balanced",
    random_state=42
)

models = {
    "LogisticRegression": log_reg,
    "RandomForest": rf
}

for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    prob = model.predict_proba(X_test)[:, 1]
    print("=" * 70)
    print(name)
    print(classification_report(y_test, pred, digits=3))
    print("ROC-AUC:", round(roc_auc_score(y_test, prob), 3))
    print("混淆矩阵:\n", confusion_matrix(y_test, pred))

LogisticRegression
              precision    recall  f1-score   support

           0      1.000     0.969     0.984       360
           1      0.686     1.000     0.814        24

    accuracy                          0.971       384
   macro avg      0.843     0.985     0.899       384
weighted avg      0.980     0.971     0.974       384

ROC-AUC: 0.998
混淆矩阵:
 [[349  11]
 [  0  24]]


RandomForest
              precision    recall  f1-score   support

           0      0.992     0.997     0.994       360
           1      0.955     0.875     0.913        24

    accuracy                          0.990       384
   macro avg      0.973     0.936     0.954       384
weighted avg      0.989     0.990     0.989       384

ROC-AUC: 1.0
混淆矩阵:
 [[359   1]
 [  3  21]]


### 2.1 小结

从结果看，IsolationForest 在没有使用欺诈标签训练的情况下，已经能识别一部分异常交易，说明无监督异常检测适合冷启动场景。但它的缺点是阈值和污染比例较难设定，容易误报或漏报。

有监督模型利用了标签信息，因此在测试集上的表现通常更稳定。LogisticRegression 可解释性更强，RandomForest 能捕捉非线性关系，通常效果更好。实际系统中，可以先用无监督方法做冷启动筛查，积累人工审核标签后，再逐步训练有监督模型。

### 2.2 行为序列统计

下面构造一个 240 行的用户行为日志，包含 30 个用户，每个用户 8 个行为。然后统计相邻行为对的频数与转移概率。

In [4]:
from collections import Counter, defaultdict

np.random.seed(7)

actions = ["view_home", "view_product", "add_to_cart", "checkout", "leave", "search", "view_coupon"]

# 设定一个偏电商场景的转移概率，用于生成更真实的模拟日志
transition = {
    "view_home":     [0.05, 0.35, 0.02, 0.00, 0.18, 0.30, 0.10],
    "search":        [0.03, 0.55, 0.04, 0.00, 0.18, 0.15, 0.05],
    "view_product": [0.03, 0.25, 0.25, 0.02, 0.25, 0.12, 0.08],
    "view_coupon":  [0.02, 0.30, 0.28, 0.10, 0.20, 0.05, 0.05],
    "add_to_cart":  [0.01, 0.15, 0.10, 0.45, 0.20, 0.04, 0.05],
    "checkout":     [0.05, 0.10, 0.02, 0.08, 0.70, 0.03, 0.02],
    "leave":        [0.55, 0.15, 0.00, 0.00, 0.20, 0.08, 0.02],
}

records = []
base_time = pd.Timestamp("2026-05-01 09:00:00")

for user_idx in range(1, 31):
    user_id = f"U{user_idx:03d}"
    current_action = np.random.choice(["view_home", "search"])
    current_time = base_time + pd.Timedelta(minutes=np.random.randint(0, 5000))
    
    for step in range(8):
        records.append({
            "user_id": user_id,
            "timestamp": current_time,
            "action": current_action
        })
        probs = transition[current_action]
        current_action = np.random.choice(actions, p=probs)
        current_time += pd.Timedelta(minutes=np.random.randint(1, 25))

logs = pd.DataFrame(records).sample(frac=1, random_state=7).reset_index(drop=True)
print("日志行数:", len(logs))
print("用户数:", logs["user_id"].nunique())
logs.head(10)

日志行数: 240
用户数: 30


,user_id,timestamp,action
0,U011,2026-05-02 21:31:00,search
1,U017,2026-05-02 22:57:00,leave
2,U001,2026-05-02 05:47:00,view_coupon
3,U026,2026-05-03 18:48:00,add_to_cart
4,U019,2026-05-03 00:55:00,view_product
5,U024,2026-05-02 22:35:00,leave
6,U012,2026-05-04 03:33:00,view_product
7,U012,2026-05-04 02:01:00,view_home
8,U016,2026-05-02 16:31:00,search
9,U007,2026-05-02 17:56:00,view_product


In [5]:
# 按 user_id 和 timestamp 排序，并得到每个用户的行为序列
logs_sorted = logs.sort_values(["user_id", "timestamp"])
user_sequences = logs_sorted.groupby("user_id")["action"].apply(list)

user_sequences.head()

user_id
U001    [search, leave, search, view_coupon, view_prod...
U002    [search, leave, view_home, view_home, search, ...
U003    [view_home, view_product, add_to_cart, checkou...
U004    [search, leave, leave, leave, view_home, searc...
U005    [search, leave, view_product, view_product, le...
Name: action, dtype: object

In [6]:
# 统计所有用户的相邻行为对频数
pair_counter = Counter()
from_action_counter = Counter()

for seq in user_sequences:
    for a, b in zip(seq[:-1], seq[1:]):
        pair_counter[(a, b)] += 1
        from_action_counter[a] += 1

transition_df = pd.DataFrame([
    {
        "from_action": a,
        "to_action": b,
        "count": cnt,
        "transition_prob": cnt / from_action_counter[a]
    }
    for (a, b), cnt in pair_counter.items()
]).sort_values(["from_action", "transition_prob"], ascending=[True, False]).reset_index(drop=True)

transition_df.head(20)

,from_action,to_action,count,transition_prob
0,add_to_cart,checkout,10,0.625000
1,add_to_cart,add_to_cart,2,0.125000
2,add_to_cart,view_coupon,2,0.125000
3,add_to_cart,view_product,1,0.062500
4,add_to_cart,leave,1,0.062500
5,checkout,leave,10,0.769231
6,checkout,search,1,0.076923
7,checkout,checkout,1,0.076923
8,checkout,view_product,1,0.076923
9,leave,view_home,16,0.444444


In [7]:
# 每个起始行为最可能转移到哪里
most_likely_transition = (
    transition_df.sort_values(["from_action", "transition_prob"], ascending=[True, False])
    .groupby("from_action")
    .head(1)
    .reset_index(drop=True)
)

most_likely_transition

,from_action,to_action,count,transition_prob
0,add_to_cart,checkout,10,0.625000
1,checkout,leave,10,0.769231
2,leave,view_home,16,0.444444
3,search,view_product,21,0.477273
4,view_coupon,view_product,5,0.357143
5,view_home,search,12,0.352941
6,view_product,add_to_cart,15,0.283019


### 2.2 行为转移解释

我认为比较有意思的两类行为转移是：

1. **`view_product → leave`**：如果用户浏览商品后直接离开，说明商品详情页可能没有解决用户的核心疑问，例如价格、评价、配送、优惠信息不够清楚。产品优化上可以重点改进商品页信息结构，例如把优惠券、评价摘要、物流时效放到更明显的位置。

2. **`view_coupon → add_to_cart` 或 `add_to_cart → checkout`**：如果查看优惠券后加入购物车、加入购物车后结算的概率较高，说明优惠信息和购物车结算流程对转化有明显影响。产品可以在用户犹豫时展示合理优惠，但也要避免过度打扰用户。

## 3. 思考题

### （1）无监督评估策略

如果一开始完全没有标注，我会把“账号异常行为检测”系统当成一个逐步迭代的风险筛查系统，而不是一开始就追求精确分类。可以从以下几种思路评估模型是否还不错。

**第一，抽样人工审核。** 先让模型输出异常分数，按照分数从高到低抽取一批账号，再从中低分段随机抽取一批账号，交给人工审核。这样可以估计高风险样本中的真实异常比例，也可以观察模型是否只是在抓一些无意义的噪声。优点是直接、可信，可以快速形成第一批标签；缺点是人工成本高，而且审核标准可能不一致。

**第二，人工构造异常或半合成异常。** 根据业务经验设计一些明显异常行为，例如短时间高频登录、频繁切换设备、异地登录后批量操作、异常时间段连续访问敏感页面等，把这些行为注入正常日志中，观察模型能否识别。优点是成本低、可控，适合冷启动阶段测试模型有没有基本能力；缺点是构造出来的异常可能过于理想化，和真实攻击行为有差距。

**第三，历史规则或业务指标对照。** 如果网站已有一些简单风控规则，例如同一 IP 批量注册、短时间连续失败登录，可以用这些规则命中的账号作为弱标签，检查模型异常分数是否更高。也可以观察被模型判为高风险的账号，在后续是否更容易出现投诉、封禁、退款、异常登录等结果。优点是利用现有数据，比较容易落地；缺点是规则本身不一定准确，可能把旧系统偏差带入新模型。

**第四，小流量 AB 测试。** 对一小部分流量启用模型干预，例如增加验证码、二次验证或人工复核，观察异常损失是否下降、正常用户体验是否受影响。优点是最接近真实业务效果；缺点是需要谨慎控制风险，不能一开始就对大量用户强干预。

综合来说，我会先用人工审核和半合成异常验证模型，再用弱标签和后续业务结果持续校准，最后在小流量中做 AB 测试。

### （2）序列模式挖掘：高完成率学生的典型学习行为模式分析方案

要挖掘在线学习平台中“高完成率学生”的典型学习行为模式，我会先把原始行为日志整理成以学生为单位的行为序列。具体做法是：先清洗日志，去除无效记录和重复点击，统一行为名称，例如把“播放视频”“继续播放”归为“看视频”，把“提交练习”“完成测验”归为“做题”。然后按照 `user_id` 分组，并在每个学生内部按照 `timestamp` 排序，得到类似 `登录 → 看视频 → 做题 → 查看解析 → 讨论 → 继续看视频` 的行为序列。为了避免序列过长，也可以按课程章节、学习日或一次登录 session 切分。

接着定义和统计常见模式。最简单的方法是统计相邻二元行为对和三元行为片段，例如 `看视频 → 做题`、`做题 → 查看解析`、`看视频 → 做题 → 查看解析`。可以计算每种模式的出现次数、支持度和转移概率。进一步可以参考关联规则挖掘思想，把一个学习 session 看成一个事务，分析哪些行为组合经常共同出现，例如 `{看视频, 做题, 查看解析}` 是否经常出现。对于有顺序要求的模式，则可以使用序列模式挖掘方法，统计满足顺序关系的高频子序列。

然后把这些模式与最终完成率结合起来。可以先按照完成率把学生分成高完成率组和低完成率组，例如完成率前 30% 为高完成率组，后 30% 为低完成率组。分别统计两组中的行为模式频率，再比较差异。如果某些模式在高完成率组中显著更常见，而在低完成率组中很少出现，就可以认为它们可能是“成功学习路径”的候选模式。例如，如果高完成率学生更常出现 `看视频 → 做题 → 查看解析 → 再做题`，说明他们不是只被动看视频，而是会通过练习和反馈形成闭环。

最后可以把结果转化为产品建议。比如当系统发现某学生连续看视频但很少做题时，可以提醒他完成配套练习；当学生做错题后没有查看解析时，可以推荐解析或讨论区内容。这样，序列模式挖掘不仅能描述学生行为，还能帮助平台设计更有效的学习引导策略。